In [13]:
#imports 
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import balanced_accuracy_score
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt

## Importing Data:

In [14]:
comp_data_path = '/kaggle/input/competitions/playground-series-s6e4/'
orig_path = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv'

train_df = pd.read_csv(comp_data_path + 'train.csv')
print('Train data: ', train_df.shape)
display(train_df.head())

orig_df = pd.read_csv(orig_path)
print('Orignal data: ', orig_df.shape)
display(orig_df.head())

test_df = pd.read_csv(comp_data_path + 'test.csv')
print('Test data: ', test_df.shape)
display(test_df.head())

Train data:  (630000, 21)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Orignal data:  (10000, 20)


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


Test data:  (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [15]:
target = 'Irrigation_Need'

X = train_df.drop(columns=['id', target], axis=1)
X_orig = orig_df.drop(target, axis=1)
X_test = test_df.drop('id', axis=1)

y = train_df[target]
y_orig = orig_df[target]

In [16]:
X_combined = pd.concat([X, X_orig], axis=0, ignore_index=True)
y_combined = pd.concat([y, y_orig], axis=0, ignore_index=True).map({'Low': 0, 'Medium': 1, 'High': 2})

print('X_combined: ', X_combined.shape)
print('y_combined: ', y_combined.shape)

X_combined:  (640000, 19)
y_combined:  (640000,)


## Feature Engineering

In [17]:
num_cols = list(X_combined.select_dtypes(include='number').columns)
cat_cols = list(X_combined.select_dtypes("object").columns)

In [18]:
M = X_combined[num_cols].max()

In [19]:
#thresholds for num features
Soil_Moisture_thresh = 25
Rainfall_mm_thresh = 300
Temperature_C_thresh = 30
Wind_Speed_kmh_thresh = 10

In [20]:
def binary_magic_features(df):
    
    df["soil_25"]      = (df["Soil_Moisture"]       < Soil_Moisture_thresh).astype(int)
    df["rain_300"]     = (df["Rainfall_mm"]         < Rainfall_mm_thresh).astype(int)
    df["temp_30"]      = (df["Temperature_C"]       > Temperature_C_thresh).astype(int)
    df["wind_10"]      = (df["Wind_Speed_kmh"]      > Wind_Speed_kmh_thresh).astype(int)  
    df["is_harvest"]   = (df["Crop_Growth_Stage"]  == "Harvest").astype(int)
    df["is_sowing"]    = (df["Crop_Growth_Stage"]  == "Sowing").astype(int)
    df["mulching_yes"] = (df["Mulching_Used"]      == "Yes").astype(int)
    return df


def magic_score(df):
    
    high = 2 * df['soil_25'] + 2 * df['rain_300'] + df['temp_30'] + df['wind_10']
    low = 2 * df['is_harvest'] + 2 * df['is_sowing'] + df['mulching_yes']
    df['magic_score'] = high - low
    return df


def digit_decomp(df):
    
    for c in num_cols:
        for k in range(-4,4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')

        if M[c]<10:
            df[c]=df[c].round(3)
        elif M[c]<100:
            df[c]=df[c].round(2)
        else:
            df[c]=df[c].round(1)
    return df 



def create_features(df):
    df = binary_magic_features(df)
    df = digit_decomp(df)
    df = magic_score(df)
    return df

In [21]:
X_combined = create_features(X_combined)
X_test = create_features(X_test)

/tmp/ipykernel_55/3531305106.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')
/tmp/ipykernel_55/3531305106.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')
/tmp/ipykernel_55/3531305106.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

In [22]:
drop_cols=[c for c in X_test.columns if X_test[c].nunique()==1]
print(f"drop_cols:{drop_cols}")

X_combined.drop(drop_cols,axis=1,inplace=True)
X_test.drop(drop_cols,axis=1,inplace=True)

drop_cols:['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']


In [23]:
print((X_combined.shape[1] - train_df.shape[1]),' new features created')

72  new features created


### Logit-score features
We pre-fit LogisticRegression on the orignal dataset which gets a ~100% accuracy there, then we hardcode the coefficients learned directly here are constants to make logit features....

So, logit score is continuous probability (0.0 to 1.0)
                 fine-grained signal per class
                 captures the SAME rules but with
                 learned optimal weights per feature

In [24]:
logit_coefs= {
        "Low": {
            "intercept": 16.3173, "soil_25": -11.0237, "temp_30": -5.8559,
            "rain_300": -10.8500, "wind_10": -5.8284,
            "is_flowering": -5.4155, "is_harvest": 5.5073, "is_sowing": 5.2299, "is_vegetative": -5.4617,
             "mulching_yes": 2.8613,
        },
        "Medium": {
            "intercept": 4.6524, "soil_25": 0.3290, "temp_30": -0.0204,
            "rain_300": 0.1542, "wind_10": 0.0841,
            "is_flowering": 0.3586, "is_harvest": -0.1348, "is_sowing": -0.3547, "is_vegetative": 0.3334,
            "mulching_yes": 0.0142,
        },
        "High": {
            "intercept": -20.9697, "soil_25": 10.6947, "temp_30": 5.8763,
            "rain_300": 10.6958, "wind_10": 5.7444,
            "is_flowering": 5.0569, "is_harvest": -5.3725, "is_sowing": -4.8752, "is_vegetative": 5.1283,
            "mulching_yes": -2.8755,
        },
    }

In [25]:
def add_logit_scores(df):
    
    # Binary flags needed
    flags = {
        'is_flowering' : (df['Crop_Growth_Stage'] == 'Flowering').astype(float),
        'is_harvest'   : (df['Crop_Growth_Stage'] == 'Harvest').astype(float),
        'is_sowing'    : (df['Crop_Growth_Stage'] == 'Sowing').astype(float),
        'is_vegetative': (df['Crop_Growth_Stage'] == 'Vegetative').astype(float),
        'mulching_yes' : (df['Mulching_Used']     == 'Yes').astype(float),
        'soil_25': df['soil_25'].astype(float),
        'temp_30': df['temp_30'].astype(float),
        'rain_300':df['rain_300'].astype(float),
        'wind_10': df['wind_10'].astype(float),
    }

    # For each class compute the logit (linear combination)
    for cls, coefs in logit_coefs.items():
        logit = coefs['intercept']
        for feat, val in flags.items():
            logit = logit + coefs[feat] * val
        
        # Store raw logit AND sigmoid probability
        df[f'logit_{cls}']    = logit
        df[f'logprob_{cls}']  = 1 / (1 + np.exp(-logit))
    
    return df

In [26]:
X_combined = add_logit_scores(X_combined)
X_test = add_logit_scores(X_test)

### Feature crosses

In [27]:
def feature_crosses(X):
        X = X.copy()

        # Convert to string first
        cat_cols = X.select_dtypes(include=['category', 'object']).columns
        X[cat_cols] = X[cat_cols].astype(str)
        
        # Create _orig copies
        for col in cat_cols:
            X[f'{col}_orig'] = X[col]
        
        # Numerical crosses
        X['heat_humidity_stress'] = X['Temperature_C'] * (1 - X['Humidity']/100)
        X['water_availability']   = X['Rainfall_mm'] + X['Previous_Irrigation_mm']
        X['evaporation_proxy']    = X['Temperature_C'] * X['Wind_Speed_kmh'] * X['Sunlight_Hours']
        X['soil_health']          = X['Organic_Carbon'] * X['Soil_Moisture']

        #Cat x Cat
        X['region_season']     = X['Region'].astype(str) + '_' + X['Season'].astype(str)
        X['crop_growth']       = X['Crop_Type'].astype(str) + '_' + X['Crop_Growth_Stage'].astype(str)
        X['soil_crop']         = X['Soil_Type'].astype(str) + '_' + X['Crop_Type'].astype(str)
        X['source_irrigation'] = X['Water_Source'].astype(str) + '_' + X['Irrigation_Type'].astype(str)
        
        # Season cyclical
        season_map = {'Spring':0, 'Summer':1, 'Autumn':2, 'Winter':3}
        X['season_num'] = X['Season'].astype(str).map(season_map)
        X['season_sin'] = np.sin(2 * np.pi * X['season_num'] / 4)
        X['season_cos'] = np.cos(2 * np.pi * X['season_num'] / 4)
        X.drop('season_num', axis=1, inplace=True)

        #Domain Specific
        X['water_stress'] = (X['Temperature_C'] * X['Wind_Speed_kmh']) / (X['Rainfall_mm'] + X['Soil_Moisture'] + 1) # Water stress index 
        X['crop_demand']  = X['Temperature_C'] * X['Sunlight_Hours'] * (1 - X['Humidity']/100) # Crop water demand proxy
        X['retention']    = X['Soil_Moisture'] * X['Organic_Carbon'] * X['Electrical_Conductivity'] # Soil water retention
        X['irr_efficiency'] = X['Previous_Irrigation_mm'] / (X['Field_Area_hectare'] + 1) # Irrigation efficiency
        X['rain_per_area'] = X['Rainfall_mm'] / (X['Field_Area_hectare'] + 1)
    
        return X

In [28]:
X_combined = feature_crosses(X_combined)
X_test = feature_crosses(X_test)

## Model

In [29]:
cat_cols = X_combined.select_dtypes('object').columns

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        # ('num', StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

In [31]:
X_processed = preprocessor.fit_transform(X_combined)
X_test_processed = preprocessor.transform(X_test)

In [33]:
xgb_params = {
    'learning_rate': 0.03, 
    'max_depth': 4, 
    'min_child_weight': 2,
    'subsample': 0.98,
    'gamma': 4.2, 
    'colsample_bytree': 0.5, 
    'lambda': 0.001,
    'alpha': 4.1,
    'n_estimators': 50_000, 
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'device': 'cuda',
    'num_class': 3
}

In [34]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
bas_scores = []
n_classes = y_combined.nunique()
oof_test_m2 = np.zeros((X_test_processed.shape[0], n_classes))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_processed, y_combined)):
    X_tr, X_val = X_processed[tr_idx], X_processed[val_idx]
    y_tr, y_val = y_combined.iloc[tr_idx], y_combined.iloc[val_idx]

    sw = compute_sample_weight(class_weight="balanced", y=y_tr)
    
    model = XGBClassifier(
        **xgb_params,
        random_state=42,
        
        callbacks=[
            xgb.callback.EarlyStopping( #early stopping callback
                rounds=200,
                save_best=True,
            )
        ]
    )

    model.fit(
        X_tr,
        y_tr,
        sample_weight=sw,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)

    test_preds = model.predict_proba(X_test_processed)
    oof_test_m2 += test_preds / skf.n_splits

    bas = balanced_accuracy_score(y_val, preds)
    bas_scores.append(bas)
    print(f"Fold {fold} BAS: {bas}")
print(f"Mean BAS: {np.mean(bas_scores)}")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [06:16:51] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 0 BAS: 0.9732687626031377
Fold 1 BAS: 0.9725492265655942
Fold 2 BAS: 0.9724371659681893
Fold 3 BAS: 0.9733387565117368
Fold 4 BAS: 0.9690607073351906
Mean BAS: 0.9721309237967697


In [35]:
preds = oof_test_m2.argmax(axis=1)
preds = pd.Series(preds).map({0: "Low", 1: "Medium", 2: "High"})

submission = pd.DataFrame({
    "id": test_df["id"],
    target: preds
})

submission.to_csv("submission_logit_score.csv", index=False)